# Segmentation pipeline debug

Visualizes **every stage** of the production pipeline on a sample (default: **1 image per class**):

| Stage | Panels |
|-------|--------|
| Preprocess | grayscale → CLAHE work → opened background |
| Masks | text_density, dark_luma, black_hat, glare, edge_density, combined |
| Detection | raw candidates → kept (scoring) → final labels → overlay |
| Output | crops + on-disk bundle gallery |

**Kernel:** `make setup` then `uv run jupyter notebook debug.ipynb` → **Python (pdiseg)**.
After code changes: **Restart Kernel** and run all cells.

CLI: `make debug`

## 1. Setup

In [ ]:
%matplotlib inline

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path('.').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Reload autoreload without warning when already active.
_ip = get_ipython()
if _ip is not None:
    try:
        _ip.run_line_magic('reload_ext', 'autoreload')
    except Exception:
        _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')

from pdiseg.debug.reload import assert_pipeline_schema, reload_pipeline_modules

reload_pipeline_modules()
assert_pipeline_schema()

import pdiseg
from pdiseg.debug.sample import (
    DebugFrameResult,
    DebugFrameView,
    build_sample_views,
)
from pdiseg.debug.viz import (
    list_bundle_artifacts,
    plot_all_crops,
    plot_all_detection_stage,
    plot_all_mask_layers,
    plot_all_preprocess_stages,
    plot_bundle_gallery,
    plot_frame_full_debug,
    plot_frame_pipeline,
    scored_table,
)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.max_open_warning'] = 80


def show(fig):
    """Display figure inline (works in Cursor / Jupyter)."""
    if fig is not None:
        display(fig)
        plt.close(fig)


def show_all(figs):
    for fig in figs:
        show(fig)


DATASET = ROOT / 'data' / 'Train_and_Validation'
DEBUG_RESULT = ROOT / 'debug_result'
PER_CLASS = 1

if not DATASET.is_dir():
    raise FileNotFoundError(f'Dataset not found: {DATASET}')

print(f'pdiseg: {pdiseg.__file__}')
print(f'Classes: {len(pdiseg.list_classes(DATASET))}')
print(f'Sample: up to {PER_CLASS} image(s) per class')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
pdiseg: /home/felipe-versiane/code/academic/9/trabalho-PDI/src/pdiseg/__init__.py
Classes: 18
Sample: up to 1 image(s) per class


## 2. Run pipeline + build views

In [ ]:
sample_report = pdiseg.run_debug_sample(
    DATASET,
    DEBUG_RESULT / 'result',
    bundle_root=DEBUG_RESULT / 'bundles',
    per_class=PER_CLASS,
)

sample_views = build_sample_views(sample_report)
images = [view.image for view in sample_views]
prep_items = [
    (f"{view.item.class_name.split('_', 1)[0]}\n{view.item.source.name[:22]}", view.snapshot)
    for view in sample_views
]

ds = sample_report.dataset_report
print(f'frames={ds.total_frames} crops={ds.total_crops} empty={ds.empty_frames}')
print(f'avg crops/frame={ds.avg_crops_per_frame:.2f}')
print(f'result  -> {sample_report.result_root}')
print(f'bundles -> {sample_report.bundle_root}')

pd.DataFrame(
    [
        {
            'class': view.item.class_name,
            'file': view.item.source.name,
            'candidates': len(view.snapshot.detection.candidates),
            'kept': len(view.snapshot.detection.kept),
            'labels': len(view.snapshot.detection.labels),
            'bundle_files': len(list_bundle_artifacts(view.item.bundle_dir, view.item.source.stem)),
        }
        for view in sample_views
    ]
)

TypeError: CandidateMasks.__init__() got an unexpected keyword argument 'dog_text'

## 3. Preprocess — grayscale, CLAHE work, opened background

Three grids (one per sub-stage), all sample frames.

In [ ]:
show_all(plot_all_preprocess_stages(prep_items))

## 4. Masks — every filter layer

One grid per mask: `text_density`, `dark_luma`, `black_hat`, `glare`, `edge_density`, `combined`.

In [ ]:
show_all(plot_all_mask_layers(prep_items))

## 5. Detection stages — candidates → kept → labels → overlay

Red = raw connected components, yellow = kept after scoring, green = final labels.

In [ ]:
for stage in ('candidates', 'kept', 'labels', 'overlay'):
    show(plot_all_detection_stage(images, prep_items, stage=stage))

## 6. Saved crops — all sample frames

In [ ]:
titles = [t for t, _ in prep_items]
labels = [view.item.labels for view in sample_views]
show(plot_all_crops(images, labels, titles))

## 7. Full pipeline — 8-panel summary (every sample frame)

In [ ]:
for view in sample_views:
    item = view.item
    title = f"{item.class_name} / {item.source.name}"
    show(plot_frame_pipeline(view.image, view.snapshot, title=title))

## 8. Extended debug — all masks + all stages (every sample frame)

3-row panel: preprocess → every mask layer → candidates / kept / labels / overlay.

In [ ]:
for view in sample_views:
    item = view.item
    title = f"{item.class_name} / {item.source.name}"
    show(plot_frame_full_debug(view.image, view.snapshot, title=title))

## 9. Scoring — top candidates per frame

In [ ]:
score_rows = []
for view in sample_views:
    item = view.item
    for row in scored_table(view.snapshot.detection.scored)[:8]:
        score_rows.append(
            {
                'class': item.class_name,
                'file': item.source.name,
                **{k: v for k, v in row.items() if k != 'box'},
                'box': row['box'],
            }
        )
pd.DataFrame(score_rows)

## 10. On-disk bundles — every saved PNG per frame

Gallery of artifacts written by the pipeline: source, masks, candidates, kept, labels, crops, pipeline panel.

In [ ]:
for view in sample_views:
    item = view.item
    show(plot_bundle_gallery(item.bundle_dir, item.source.stem))